# Nextbike Brno Availability Baseline

Generated at `2026-05-25T02:09:59+02:00`.

This notebook reproduces the first 30-minute station emptiness baseline/model analysis.


## Current Summary

| Metric | Value |
|---|---|
| `collection_runs` | `353` |
| `first_collected_at` | `2026-05-24 18:32:12+02:00` |
| `latest_collected_at` | `2026-05-25 02:09:06+02:00` |
| `station_rows` | `104841` |
| `free_bike_rows` | `184180` |
| `distinct_stations` | `297` |
| `bikes_available_min_avg_max` | `553/566.28/572` |
| `db_size_mb` | `13.51` |

## Dataset Summary

| Metric | Value |
|---|---|
| `rows` | `94446` |
| `stations` | `297` |
| `first_collected_at` | `2026-05-24 20:17:28+02:00` |
| `latest_collected_at` | `2026-05-25 01:38:47+02:00` |
| `empty_future_rate` | `0.3955` |
| `avg_abs_change` | `0.0832` |
| `changed_rate` | `0.0688` |
| `null_lag_5m` | `2970` |
| `null_lag_15m` | `6831` |

Build settings:

- horizon minutes: `30`
- max target delay minutes: `40`
- lag tolerance minutes: `10`


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'nextbike.duckdb').exists():
    ROOT = ROOT.parent
DB_PATH = ROOT / 'data' / 'nextbike.duckdb'
TABLE_NAME = 'training_station_availability'

con = duckdb.connect(str(DB_PATH), read_only=True)
df = con.sql(f"select * from {TABLE_NAME} order by collected_at, station_id").df()
df.head()

In [ ]:
df[['bikes_now', 'bikes_future', 'empty_now', 'empty_future']].describe(include='all')

## Baseline Metrics

| Model | Accuracy | Precision | Recall | F1 | ROC AUC | Avg precision | Pred empty rate |
|---|---:|---:|---:|---:|---:|---:|---:|
| `persistence_empty_now` | 0.9896 | 0.9821 | 0.9913 | 0.9867 | 0.9900 | 0.9769 | 0.3901 |
| `station_prior_empty_rate` | 0.9559 | 0.9236 | 0.9656 | 0.9442 | 0.9841 | 0.9626 | 0.4040 |
| `low_inventory_le_1` | 0.8033 | 0.6627 | 1.0000 | 0.7972 | 0.9929 | 0.9793 | 0.5832 |
| `majority_train_class` | 0.6135 | 0.0000 | 0.0000 | 0.0000 | 0.5000 | 0.3865 | 0.0000 |

In [ ]:
from nextbike_analysis.modeling import evaluate_baselines

baseline = evaluate_baselines(
    db_path=DB_PATH,
    table_name=TABLE_NAME,
    test_fraction=0.25,
    low_bike_threshold=1,
)
[(row.name, row.f1, row.roc_auc) for row in baseline.metrics]

## Model Metrics

| Model | Accuracy | Precision | Recall | F1 | ROC AUC | Avg precision | Pred empty rate |
|---|---:|---:|---:|---:|---:|---:|---:|
| `hist_gradient_boosting` | 0.9900 | 0.9834 | 0.9910 | 0.9871 | 0.9944 | 0.9852 | 0.3895 |
| `logistic_regression` | 0.9851 | 0.9715 | 0.9904 | 0.9809 | 0.9915 | 0.9745 | 0.3940 |

In [ ]:
from nextbike_analysis.modeling import train_models

model_result, _ = train_models(
    db_path=DB_PATH,
    table_name=TABLE_NAME,
    test_fraction=0.25,
    model_dir=None,
)
[(row.name, row.f1, row.roc_auc) for row in model_result.metrics]

## Notes

The current sample is mostly overnight, so persistence is a very strong baseline.
Do not judge advanced model classes until we have morning/daytime demand in the dataset.
